In [1]:
# ============================================================
# Pyomo template: Multi-Period Production-Inventory with Backorders
# ============================================================
# Business story (very realistic):
# - Each period t, we can PRODUCE up to capacity.
# - Demand must be satisfied either:
#     (a) from available inventory, OR
#     (b) as backorder (late delivery) with penalty cost.
# - Inventory can be carried to next period with holding cost.
#
# Goal: minimize total cost = production + holding + backorder penalty
#
# This is a CORE building block of:
# - S&OP models
# - Safety stock / service level extensions
# - Multi-echelon planning
# - Rolling horizon control
# ============================================================

import pyomo.environ as pyo

def build_and_solve():
    # --------------------------
    # 1) DATA (small example)
    # --------------------------
    # Periods (e.g., weeks or months)
    T = [1, 2, 3, 4]

    # Demand in each period
    demand = {
        1: 80,
        2: 120,
        3: 60,
        4: 140
    }

    # Production capacity each period (units you can make)
    capacity = {
        1: 100,
        2: 100,
        3: 100,
        4: 100
    }

    # Costs (choose units like EUR/unit)
    prod_cost = 5.0        # cost per unit produced
    hold_cost = 1.0        # cost per unit held in inventory per period
    backorder_cost = 12.0  # penalty per unit of demand not met on time (late delivery)

    # Initial conditions
    init_inventory = 20
    init_backorder = 0

    # --------------------------
    # 2) MODEL
    # --------------------------
    m = pyo.ConcreteModel("ProdInvBackorder")

    # Set of periods
    m.T = pyo.Set(initialize=T, ordered=True)

    # Parameters
    m.D = pyo.Param(m.T, initialize=demand, within=pyo.NonNegativeReals)     # demand[t]
    m.CAP = pyo.Param(m.T, initialize=capacity, within=pyo.NonNegativeReals) # capacity[t]

    # --------------------------
    # 3) DECISION VARIABLES
    # --------------------------
    # x[t] = production quantity in period t
    m.x = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    # I[t] = inventory at end of period t
    m.I = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    # B[t] = backorder at end of period t (unmet demand carried forward)
    m.B = pyo.Var(m.T, domain=pyo.NonNegativeReals)

    # --------------------------
    # 4) CONSTRAINTS
    # --------------------------
    # 4.1) Capacity constraint: cannot produce more than capacity
    def cap_rule(m, t):
        return m.x[t] <= m.CAP[t]
    m.Capacity = pyo.Constraint(m.T, rule=cap_rule)

    # 4.2) Flow balance constraint (MOST IMPORTANT)
    #
    # Think of "net inventory position":
    #   Available to satisfy demand = (prev inventory + production) - (prev backorder)
    #
    # Equivalent standard formulation:
    #   Inventory_t - Backorder_t
    #     = Inventory_{t-1} - Backorder_{t-1} + Production_t - Demand_t
    #
    # This single equation is the heart of multi-period planning.
    def balance_rule(m, t):
        if t == min(T):
            # Use initial inventory/backorder for the first period
            prev_I = init_inventory
            prev_B = init_backorder
        else:
            prev_I = m.I[t-1]
            prev_B = m.B[t-1]

        # Net inventory position recursion:
        return (m.I[t] - m.B[t]) == (prev_I - prev_B) + m.x[t] - m.D[t]

    m.Balance = pyo.Constraint(m.T, rule=balance_rule)

    # Optional business rule:
    # If you want "no ending backorder" (must catch up by final period), uncomment:
    # m.EndBackorderZero = pyo.Constraint(expr=m.B[max(T)] == 0)

    # --------------------------
    # 5) OBJECTIVE
    # --------------------------
    # Total cost = sum_t (production + holding + backorder penalty)
    def obj_rule(m):
        return sum(
            prod_cost * m.x[t] +
            hold_cost * m.I[t] +
            backorder_cost * m.B[t]
            for t in m.T
        )

    m.Obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # --------------------------
    # 6) SOLVE
    # --------------------------
    # Try open-source solvers in common order:
    # - glpk (LP/MIP)
    # - cbc  (LP/MIP)
    # This is an LP model (continuous), so both work if installed.
    solver_names = ["glpk", "cbc"]

    results = None
    used_solver = None

    for s in solver_names:
        solver = pyo.SolverFactory(s)
        if solver.available():
            used_solver = s
            results = solver.solve(m, tee=False)
            break

    if results is None:
        raise RuntimeError(
            "No solver available. Install one solver, e.g.:\n"
            " - GLPK: conda install -c conda-forge glpk\n"
            " - CBC : conda install -c conda-forge coincbc\n"
        )

    # --------------------------
    # 7) PRINT RESULTS (planner-friendly)
    # --------------------------
    print(f"\nSolved with: {used_solver}")
    print("t | Demand | Prod | Inv_end | BO_end | Net(Inv-BO)")
    print("--|--------|------|---------|--------|-----------")

    for t in m.T:
        demand_t = pyo.value(m.D[t])
        prod_t   = pyo.value(m.x[t])
        inv_t    = pyo.value(m.I[t])
        bo_t     = pyo.value(m.B[t])
        net_t    = inv_t - bo_t
        print(f"{t:>1} | {demand_t:>6.0f} | {prod_t:>4.0f} | {inv_t:>7.0f} | {bo_t:>6.0f} | {net_t:>9.0f}")

    print(f"\nTotal cost = {pyo.value(m.Obj):.2f}\n")

    return m

if __name__ == "__main__":
    build_and_solve()



Solved with: glpk
t | Demand | Prod | Inv_end | BO_end | Net(Inv-BO)
--|--------|------|---------|--------|-----------
1 |     80 |   80 |      20 |      0 |        20
2 |    120 |  100 |       0 |      0 |         0
3 |     60 |  100 |      40 |      0 |        40
4 |    140 |  100 |       0 |      0 |         0

Total cost = 1960.00

